# High-Frequency Contact (HFC) Optimization for GōMartini 3
### A practical tutorial for building an optimized contact map from AA-MD simulations

## Introduction

Structure-based coarse-grained models such as GōMartini 3 rely on predefined native contact maps to stabilize the target structure. Traditionally, these contact maps are derived from a single static experimental structure (e.g., X-ray or cryo-EM), which may limit conformational flexibility and bias the energy landscape toward one specific conformation.

To address this limitation, we implemented a **High-Frequency Contact (HFC)** strategy in which native contacts are extracted from atomistic molecular dynamics (AA-MD) trajectories and filtered according to their persistence over time. A contact is classified as HFC if it appears in at least 70% of the analyzed frames.

This protocol improves conformational sampling while preserving structural stability in large protein assemblies.

The methodological framework described here is based on:

Olivos-Ramírez, G. E. et al.  
*An optimized contact map for GōMartini 3 enabling conformational changes in protein assemblies.*  
bioRxiv (2025).  
https://doi.org/10.1101/2025.11.14.688437

## Software Requirements

This workflow requires:

- martinize2 
- DSSP
- Python ≥ 3.9
- MDAnalysis
- numpy
- tqdm

This project was tested with:

vermouth == 0.15.0  
mkdssp == 3.1.4


## Overview of the HFC Protocol

The complete pipeline consists of five main steps:

1. Extract frames from an AA trajectory.
2. Compute residue–residue Cα contact maps for each frame.
3. Filter contacts based on a distance cutoff (typically 0.3–1.1 nm).
4. Compute contact frequency across frames and select HFCs (≥ 70%).
5. Generate the final GōMartini 3 topology using martinize2 with the optimized contact map.

This workflow is automated through two scripts:

- traj_to_pdb.py
- contact_freq.py

## Script: traj_to_pdb_v2.py

This script extracts frames from one or multiple AA trajectories and converts them into cleaned PDB files suitable for contact-map calculation.

Key features:

- Residue renumbering per chain
- Automatic chain ID assignment
- Histidine normalization (HIE/HID/HSD/HSE/HSP → HIS)
- Cysteine normalization (CYX → CYS)
- Atom name correction (ILE CD → CD1)
- Removal of problematic atoms (CY, OY, NT, CAY, CAT)


In [ ]:
## Example Usage for a MD simulation obtained with amber
!python traj_to_pdb.py \
  --trajectory strip.rep1_fit.nc \
  --topology strip.parm7 \
  --ranges 1-1121,1122-2242,2243-3364 \
  --outdir ./ \
  --stride 1

## Script: traj_to_pdb_v2.py

This script:

1. Computes contact maps for each frame.
2. Filters contacts based on distance.
3. Classifies contacts as intra-chain, inter-chain, or both (after testing we determined the best configuration is both).
4. Computes contact frequencies.
5. Selects HFC contacts (frequency ≥ threshold).
6. Identifies the frame containing the highest number of HFCs.
7. Runs martinize2 using the optimized contact map.

**Key Parameters**

--cm           Path to contact_map executable  
--type         intra | inter | both  
--threshold    Frequency cutoff (0.7 = 70%)  
--merge        Chain merging option (e.g., all or A,B,C)  
--dssp         Path to mkdssp  
--go-eps       Go potential epsilon (e.g., 15)  
--cpus         Number of CPUs  


In [ ]:
## Example Usage

!python contact_freq.py \
  --cm ./ \
  --type both \
  --cpus 15 \
  --threshold 0.7 \
  --merge all \
  --dssp mkdssp \
  --go-eps 15 \
  --from amber

## Expected Outputs

Per-frame files:
- frame_####.pdb
- frame_####.map
- filtered_frame_####.txt

Frequency analysis:
- normalized_both.txt
- high_both.txt
- high_counts_per_frame.txt

Final GōMartini3 outputs:
- topol.top (topology file)
- frane*_CG.pdb (frame of refence to start the CG simulation)
- ITP files (go_nbparms.itp, go_atomtypes.itp, and molecule_0.itp)
- run.log (log file with the parameters used for the HFC calculation)


## Interpretation

The HFC approach removes transient, low-frequency contacts while preserving structurally persistent interactions. Compared to static contact maps derived from a single structure, this method enables:

- Increased conformational flexibility
- Reduced artificial over-stabilization
- Improved large-scale motions in oligomeric assemblies

For detailed theoretical background and validation benchmarks, refer to:

Olivos-Ramírez et al., bioRxiv (2025)
https://doi.org/10.1101/2025.11.14.688437
